# **This notebook implements a EEG Auditory Attention Detectiom (AAD) system using Domain Adversarial Learning. The goal is to predict which speaker a subject is attending to while learning subject-invariant features through domain adaptation.**

# **Importing Libraries**

In [1]:
# Standard Library Imports
import os            # OS-level operations (paths, directory handling)
import math          # Mathematical functions
import shutil        # High-level file operations (copying, moving, deleting)
import random

# Data Handling & Processing
import numpy as np               # Numerical computations and array operations
import pandas as pd              # Data manipulation and analysis
import h5py                      # Handling HDF5 file formats

# Progress Visualization
from tqdm import tqdm            # Progress bars for loops

# Plotting & Visualization
import matplotlib.pyplot as plt  # Plotting and visualization tools


# PyTorch Machine Learning Stack
import torch                     # Core PyTorch library
import torch.nn as nn            # Neural network layers and utilities
import torch.optim as optim      # Optimization algorithms (SGD, Adam, etc.)
from torch.utils.data import (   # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)

# **Function to load h5 files**

In [2]:
def load_h5_dataset(file_path):
    """
    Load dataset, labels, and subject IDs from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data', 'label', and 'sub_id' datasets.

    Returns
    -------
    data : np.ndarray
        The input data stored in the HDF5 file.
    label : np.ndarray
        Corresponding labels for each data sample.
    subjects : np.ndarray
        Subject IDs associated with each data sample.
    """

    # Open the .h5 file in read-only mode to ensure safe, non-destructive access
    with h5py.File(file_path, 'r') as f:
        # Load arrays stored in the HDF5 datasets
        data = np.array(f['data'])       # Main feature/data array
        label = np.array(f['label'])     # Labels or targets for each sample
        subjects = np.array(f['sub_id']) # Subject identifier for each data entry

    # Return loaded components as NumPy arrays
    return data, label, subjects


# **Define pytorch class for data**

In [3]:
class CustomDatasets(Dataset):
    """
    A PyTorch Dataset wrapper for handling data, labels, and subject IDs.

    This class allows data to be easily fed into a DataLoader for batching,
    shuffling, and parallel loading during training or evaluation.
    """

    def __init__(self, data, labels, subjects):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like
            Input feature data, typically a NumPy array.
        labels : array-like
            Integer class labels for each sample.
        subjects : array-like
            Subject IDs corresponding to each sample (useful for subject-level splits).
        """
        self.data = data
        self.labels = labels
        self.subjects = subjects

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, index):
        """
        Retrieve a single sample by index.

        Returns
        -------
        x : torch.Tensor
            Feature tensor for the given index.
        y : torch.Tensor
            Label tensor for the given index.
        s : torch.Tensor
            Subject ID tensor for the given index.
        """
        # Convert selected sample, label, and subject ID into PyTorch tensors
        x = torch.tensor(self.data[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        s = torch.tensor(self.subjects[index], dtype=torch.long)

        return x, y, s


# **Model architecture**

In [4]:
# Gradient Reversal Layer (GRL)
# Used in Domain-Adversarial Neural Networks (DANN)
# Reverses gradient during backprop to encourage domain invariance
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        """
        Forward pass: acts as identity function.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        lambd : float
            Reversal strength; used in the backward pass.
        """
        ctx.lambd = lambd                 # Store λ for backward pass
        return x.view_as(x)               # Identity operation

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: multiply gradient by -λ.
        This forces the feature extractor to learn domain-invariant features.
        """
        return grad_output.neg() * ctx.lambd, None


def grad_reverse(x, lambd):
    """Convenience wrapper for applying the GRL."""
    return GradReverse.apply(x, lambd)


# Token Embedding Module
# Converts raw EEG input into token embeddings suitable for LSTM
class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension for each time step.
        """
        super(TokenEmbedding, self).__init__()

        # First embedding layer:
        # 1 → (d_model*4) channels using temporal convolution
        self.embed_layer = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=d_model * 4,
                kernel_size=(1, 8),
                padding='same'
            ),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Second embedding layer:
        # (d_model*4) → d_model channels using spatial convolution across channels
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(
                in_channels=d_model * 4,
                out_channels=d_model,
                kernel_size=(c_in, 1),   # Span all channels
                padding='valid'
            ),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        x : (B, C, T)
        Returns token embeddings: (B, T, d_model)
        """
        x = x.unsqueeze(1)            # (B, 1, C, T)
        x = self.embed_layer(x)       # (B, d_model*4, C, T)
        x = self.embed_layer2(x)      # (B, d_model, 1, T)
        x = x.squeeze(2)              # (B, d_model, T)
        x = x.permute(0, 2, 1)        # (B, T, d_model)
        return x


# tokenembedding + LSTM + DANN Model
# - Token embedding through convolution
# - LSTM sequence model
# - Task classifier (main task)
# - Domain classifier (subject classifier) with GRL
class DARNet_LSTM_DANN(nn.Module):
    def __init__(self, c_in=32, d_model=16, hidden=64,
                 num_classes=2, num_subjects=30):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension per token.
        hidden : int
            Hidden size of the LSTM.
        num_classes : int
            Number of output task classes.
        num_subjects : int
            Number of domain classes (e.g., subjects).
        """
        super().__init__()

        # Embed raw data into feature tokens
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM for temporal feature extraction
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Task classifier (main AAD task)
        self.classifier = nn.Linear(hidden * 2, num_classes)

        # Domain classifier (subject prediction)
        # Used via gradient reversal for adversarial training
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_subjects)
        )


    def forward(self, x, lambd=0.0):
        """
        Forward pass with optional gradient reversal.

        Parameters
        ----------
        x : (B, C, T)
            Input EEG batch.
        lambd : float
            Lambda for gradient reversal (0 during evaluation).
        """
        # 1. Token embedding
        emb = self.token_embed(x)                          # (B, T, d_model)

        # 2. Sequence modeling
        features, _ = self.lstm(emb)                       # (B, T, hidden*2)

        # 3. Global average pooling across time
        feat = features.mean(dim=1)                        # (B, hidden*2)

        # ----- Task prediction -----
        logits_task = self.classifier(feat)

        # ----- Domain prediction (with gradient reversal) -----
        rev = grad_reverse(feat, lambd)                    # Reverse gradients
        logits_domain = self.domain_classifier(rev)

        return logits_task, logits_domain



# **Function for training**

In [5]:
def set_seed(seed):
    """Seed every RNG that affects this run's randomness."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def reset_weights(model):
    """Re-initialize every layer's parameters in-place, using whatever
    RNG state is currently active. Call this AFTER set_seed(seed) so the
    weights actually depend on `seed`, even though `model` was already
    constructed before this function runs."""
    for layer in model.modules():
        if hasattr(layer, "reset_parameters"):
            layer.reset_parameters()


def train_model(model, train_loader, val_loader, epochs=100, lr=5e-4,
                weight_decay=3e-4, device="cuda", seed=0):
    """
    Train a DARNet-LSTM-DANN model with adversarial domain adaptation.

    ... (same as before) ...

    seed : int
        Controls weight initialization and training-time stochasticity
        (batch shuffling order). Run this function with several different
        seeds, keeping everything else fixed, to check whether your
        results/findings are reproducible across independent training runs.
    """
    # --- Reproducibility: seed first, THEN re-init the model's weights ---
    set_seed(seed)
    reset_weights(model)

    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    criterion_task = nn.CrossEntropyLoss()
    criterion_domain = nn.CrossEntropyLoss()

    best_val_acc = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        lambd = 2 / (1 + np.exp(-10 * (epoch / epochs))) - 1

        for x, y, s in tqdm(train_loader):
            x = x.to(device)
            y = y.to(device).long().squeeze(-1)
            s = s.to(device).long().squeeze(-1)

            optimizer.zero_grad()
            logits_task, logits_domain = model(x, lambd=lambd)
            task_loss = criterion_task(logits_task, y)
            domain_loss = criterion_domain(logits_domain, s)
            loss = task_loss + 0.5 * domain_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits_task, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        scheduler.step()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y, s in tqdm(val_loader):
                x = x.to(device)
                y = y.to(device).long().squeeze(-1)
                logits_task, _ = model(x, lambd=0.0)
                loss = criterion_task(logits_task, y)
                val_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits_task, dim=1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(
            f"[seed={seed}] Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | λ={lambd:.3f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            ckpt_name = f"best_model_seed{seed}.pth"
            torch.save(model.state_dict(), ckpt_name)
            print(f"Saved new best model -> {ckpt_name}")

    print(f"[seed={seed}] Best Val Acc:", best_val_acc)

    # Log this seed's result to a running CSV 
    log_path = "seed_results.csv"
    write_header = not os.path.exists(log_path)
    with open(log_path, "a") as f:
        if write_header:
            f.write("seed,best_val_acc\n")
        f.write(f"{seed},{best_val_acc:.4f}\n")

    return model

# **Load the data into training and val loaders**

In [6]:
# Paths to training and validation HDF5 datasets
train_h5_path = '/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/train_data.h5'
val_h5_path   = '/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/val_data.h5'

# Load data from .h5 files (features, labels, subject IDs)
X_train, y_train, s_train = load_h5_dataset(train_h5_path)
X_val,   y_val,   s_val   = load_h5_dataset(val_h5_path)

# Subject ID arrays sometimes have an extra singleton dimension → remove it
s_train = s_train.squeeze()
s_val   = s_val.squeeze()

# Convert subject IDs to 0-based consecutive integer indices
# This ensures subject IDs are compatible with PyTorch operations.
unique_subjects = np.unique(np.concatenate([s_train, s_val]))     # Find all unique subject IDs
subject2idx = {sub: i for i, sub in enumerate(unique_subjects)}   # Map original ID → new index

# Apply conversion to both sets
s_train = np.array([subject2idx[s] for s in s_train])
s_val   = np.array([subject2idx[s] for s in s_val])

# Wrap datasets in PyTorch Dataset objects
# These handle indexing and formatting into PyTorch tensors.
train_dataset = CustomDatasets(X_train, y_train, s_train)
val_dataset   = CustomDatasets(X_val, y_val, s_val)

# (Optional sanity check) Print dataset shapes
print(f"Train data: {X_train.shape}, labels: {y_train.shape}")
print(f"Val data:   {X_val.shape}, labels: {y_val.shape}")

# Build DataLoaders for batching, shuffling, and parallel reading

train_loader = DataLoader(
    train_dataset,
    batch_size=128,   # Number of samples per batch
    shuffle=True,     # Shuffle training samples each epoch
    drop_last=True    # Ensures all batches are full-sized
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,    # No need to shuffle validation data
    drop_last=False   # Keep all samples
)


Train data: (171080, 32, 128), labels: (171080, 1)
Val data:   (26320, 32, 128), labels: (26320, 1)


# **Train the model**

In [7]:
# create model
model = DARNet_LSTM_DANN(d_model=8, num_subjects=len(unique_subjects))
# run the training
model = train_model(model, train_loader, val_loader, epochs=200, lr=1e-4,
                     weight_decay=3e-4, device="cuda", seed=42)

  0%|          | 0/1336 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(
100%|██████████| 206/206 [00:07<00:00, 27.53it/s]


[seed=42] Epoch 1/200 | Train Loss: 2.2347 | Train Acc: 0.6412 | Val Loss: 1.0533 | Val Acc: 0.4521 | λ=0.000
Saved new best model -> best_model_seed42.pth


100%|██████████| 206/206 [00:07<00:00, 27.04it/s]


[seed=42] Epoch 2/200 | Train Loss: 2.0809 | Train Acc: 0.7134 | Val Loss: 1.2496 | Val Acc: 0.4428 | λ=0.025


100%|██████████| 206/206 [00:07<00:00, 26.98it/s]


...

[seed=42] Epoch 198/200 | Train Loss: 1.6933 | Train Acc: 0.9759 | Val Loss: 3.2372 | Val Acc: 0.4888 | λ=1.000


100%|██████████| 206/206 [00:07<00:00, 27.06it/s]


[seed=42] Epoch 199/200 | Train Loss: 1.6935 | Train Acc: 0.9749 | Val Loss: 3.0078 | Val Acc: 0.4851 | λ=1.000


100%|██████████| 206/206 [00:07<00:00, 27.06it/s]

[seed=42] Epoch 200/200 | Train Loss: 1.7021 | Train Acc: 0.9737 | Val Loss: 2.9527 | Val Acc: 0.4957 | λ=1.000
[seed=42] Best Val Acc: 0.5303191489361702


In [8]:
# create model
model = DARNet_LSTM_DANN(d_model=8, num_subjects=len(unique_subjects))
# run the training
model = train_model(model, train_loader, val_loader, epochs=200, lr=1e-4,
                     weight_decay=3e-4, device="cuda", seed=100)

100%|██████████| 206/206 [00:07<00:00, 27.26it/s]


[seed=100] Epoch 1/200 | Train Loss: 2.2355 | Train Acc: 0.6505 | Val Loss: 1.0764 | Val Acc: 0.4676 | λ=0.000
Saved new best model -> best_model_seed100.pth


100%|██████████| 206/206 [00:07<00:00, 27.06it/s]


[seed=100] Epoch 2/200 | Train Loss: 2.0713 | Train Acc: 0.7316 | Val Loss: 1.2545 | Val Acc: 0.4620 | λ=0.025


100%|██████████| 206/206 [00:07<00:00, 27.09it/s]


...

[seed=100] Epoch 198/200 | Train Loss: 1.7182 | Train Acc: 0.9654 | Val Loss: 2.9722 | Val Acc: 0.4831 | λ=1.000


100%|██████████| 206/206 [00:07<00:00, 26.97it/s]


[seed=100] Epoch 199/200 | Train Loss: 1.7358 | Train Acc: 0.9598 | Val Loss: 2.6419 | Val Acc: 0.4737 | λ=1.000


100%|██████████| 206/206 [00:07<00:00, 27.04it/s]

[seed=100] Epoch 200/200 | Train Loss: 1.7175 | Train Acc: 0.9647 | Val Loss: 2.8973 | Val Acc: 0.4861 | λ=1.000
[seed=100] Best Val Acc: 0.5243920972644377
